# Analisis Tren Harga Komoditas Pangan di Kawasan ASEAN Berbasis Data Warehouse melalui Integrasi Data WFP dan World Bank dengan Pendekatan Star Schema
**UAS Data Warehouse 2025/2026 | S1 Sains Data UNESA**

| Fase | Keterangan |
|------|------------|
| 1    | Extract — WFP CSV + World Bank API |
| 2    | Transform — cleaning, integrasi, star schema |
| 3    | Load — feeder ke PostgreSQL/Supabase |

## 0. Setup

In [1]:
import sys
import os
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv('../.env')

print('DATABASE_URL tersedia:', bool(os.getenv('DATABASE_URL')))

DATABASE_URL tersedia: True


## 1. Extract

### 1a. WFP Dataset

Link: https://www.kaggle.com/datasets/abhishekgupta56447/global-food-prices-database-wfp

In [ ]:
import subprocess

batches_wfp = [
    ('1', '2024', '2024'),
    ('2', '2025', '2025'),
    ('3', '2024', '2025'),
]

for batch, start, end in batches_wfp:
    result = subprocess.run(
        ['python', 'scraper/reader_wfp.py', '--batch', batch, '--start', start, '--end', end],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print('ERROR:', result.stderr)

### 1b. World Bank API

In [ ]:
batches_wb = [
    ('1', '2024', '2024', 'FP.CPI.TOTL.ZG'),
    ('2', '2025', '2025', 'NY.GDP.PCAP.CD'),
    ('3', '2024', '2025', 'all'),
]

for batch, start, end, indicator in batches_wb:
    result = subprocess.run(
        ['python', 'scraper/reader_worldbank.py',
         '--batch', batch, '--start', start, '--end', end, '--indicator', indicator],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print('ERROR:', result.stderr)

### 1c. Verifikasi file raw

In [ ]:
import glob
import os

raw_files = sorted(glob.glob('data/raw/*'))
print(f'Total file di data/raw/: {len(raw_files)}')
for f in raw_files:
    size_kb = os.path.getsize(f) / 1024
    print(f'  {os.path.basename(f):50s} {size_kb:8.1f} KB')

## 2. Transform

In [ ]:
result = subprocess.run(
    ['python', 'etl/preprocess.py'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('ERROR:', result.stderr)

### Verifikasi output processed

In [ ]:
import pandas as pd

tables = ['fact_harga_pangan', 'dim_waktu', 'dim_negara', 'dim_komoditas', 'dim_indikator']

for t in tables:
    path = f'data/processed/{t}.csv'
    try:
        df = pd.read_csv(path)
        print(f'{t:30s}: {len(df):,} baris | kolom: {list(df.columns)}')
    except FileNotFoundError:
        print(f'{t:30s}: FILE TIDAK DITEMUKAN')

In [ ]:
# Preview fact table
fact = pd.read_csv('data/processed/fact_harga_pangan.csv')
print(f'Shape: {fact.shape}')
display(fact.head(10))
display(fact.describe())

## 3. Load

In [ ]:
result = subprocess.run(
    ['python', 'etl/feeder.py'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('ERROR:', result.stderr)

### Verifikasi data di PostgreSQL

In [ ]:
import psycopg2
import os

conn = psycopg2.connect(os.getenv('DATABASE_URL'))

tables = ['dim_waktu', 'dim_negara', 'dim_komoditas', 'dim_indikator', 'fact_harga_pangan']
with conn.cursor() as cur:
    for t in tables:
        cur.execute(f'SELECT COUNT(*) FROM {t}')
        count = cur.fetchone()[0]
        print(f'{t:30s}: {count:,} baris')

conn.close()